# Interactive visualization showcase

This notebook demonstrates the reusable Plotly visualization API with six
predictive-maintenance data layouts:

- C-MAPSS run-to-failure sensor trajectories;
- GFD healthy and broken-tooth vibration distributions;
- MAPM telemetry and event timelines.
- MetroPT2 compressor signals;
- CARE wind-turbine SCADA signals;
- Backblaze fleet failure rates.

Data preparation remains in Polars. Dataset-specific modules supply column
names, labels, and defaults, while the shared API returns Plotly figures.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import polars as pl
import datasets

datasets.available_plots()

## C-MAPSS sensor trajectory

The reusable time-series function filters one engine, applies the configured
point limit, and uses C-MAPSS-specific sensor labels.

In [ ]:
datasets.download("cmapss")
cmapss = datasets.load("cmapss", subset="FD001", split="train")
cmapss_figure = datasets.visualize(
    "cmapss",
    "sensor_trajectory",
    cmapss,
    entity=1,
)
cmapss_figure.show()

## GFD condition comparison

The same plotting layer renders a grouped distribution when the dataset
configuration describes a feature and condition column.

In [ ]:
datasets.download("gfd")
gfd = pl.concat([
    datasets.load("gfd", condition="healthy", load=50),
    datasets.load("gfd", condition="broken", load=50),
])
gfd_figure = datasets.visualize("gfd", "condition_distribution", gfd)
gfd_figure.show()

## MAPM telemetry and errors

MAPM demonstrates two specifications for one dataset. Telemetry is rendered as
multiple lines, while error records use an event timeline.

In [ ]:
datasets.download("mapm")
telemetry = datasets.load("mapm", table="telemetry")
telemetry_figure = datasets.visualize(
    "mapm",
    "telemetry",
    telemetry,
    entity=1,
)
telemetry_figure.show()

In [ ]:
errors = datasets.load("mapm", table="errors")
error_figure = datasets.visualize(
    "mapm",
    "error_timeline",
    errors,
    entity=1,
)
error_figure.show()

## Large, opt-in datasets

The following examples do not start downloads automatically. Download only the
dataset you need with `scripts/download.py` before running its cell. The
loaders return Polars `LazyFrame` objects so aggregation and bounded projection
happen before Plotly receives data.

### MetroPT2 compressor signals

Run `python scripts/download.py metropt2` once before this example.

In [ ]:
metropt2 = datasets.load("metropt2")
metropt2_figure = datasets.visualize(
    "metropt2",
    "sensor_signals",
    metropt2,
)
metropt2_figure.show()

### CARE wind-turbine SCADA signals

Run `python scripts/download.py care` once, then replace the recording path
below with one CSV path from the extracted collection. CARE schemas vary, so
you can pass `time_column` and `columns` explicitly when automatic selection is
not appropriate.

In [ ]:
care = datasets.load("care", recording="path/to/recording.csv")
care_figure = datasets.visualize(
    "care",
    "scada_signals",
    care,
    # time_column="timestamp",
    # columns=["signal_1_mean", "signal_2_mean"],
)
care_figure.show()

### Backblaze daily failure rate

Run `python scripts/download.py backblaze --variant 2025-q1` once before this
example. Daily aggregation is executed lazily over the quarter.

In [ ]:
backblaze = datasets.load("backblaze", variant="2025-q1")
backblaze_figure = datasets.visualize(
    "backblaze",
    "daily_failure_rate",
    backblaze,
)
backblaze_figure.show()

## Extending the showcase

Add typed specifications to `datasets/<dataset-id>/visualization.py` and
register them in `datasets/visualization/registry.py`. Extend the shared
plotting functions only when a genuinely reusable chart type is needed.